# Diagnóstico Inicial del Dataset MINEDUC

Este notebook documenta el estado inicial del conjunto de datos extraído desde el portal de MINEDUC para el nivel escolar `DIVERSIFICADO`. El objetivo es respaldar con código y tablas los hallazgos previos al plan de limpieza.

## Objetivos del diagnóstico

Este análisis cubre como mínimo:

- número de registros y variables,
- tipo de dato de cada variable,
- cantidad y porcentaje de valores faltantes por variable,
- cantidad de valores únicos,
- cantidad de registros duplicados exactos,
- variables con valores fuera de dominio o inconsistentes,
- variables con formatos inconsistentes,
- identificación de problemas potenciales de calidad de datos.

In [1]:
from __future__ import annotations

import re
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 120)

# Jupyter suele ejecutarse desde `notebooks/`; buscamos la raiz real del proyecto.
ROOT = Path.cwd().resolve()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent
RAW_PATH = ROOT / 'data' / 'raw' / 'establecimientos_diversificado_guatemala.csv'
CAT_DIR = ROOT / 'data' / 'raw' / 'catalogos'

PLACEHOLDERS = {'', '--', 'N/A', 'NULL', '-', '.', 'Sin dato', 'SIN DATO', 'NA', 'N.D.'}
METADATA_COLUMNS = [
    'fecha_extraccion',
    'fuente_url',
    'nivel_consulta',
    'sector_consulta',
    'plan_consulta',
    'modalidad_consulta',
    'departamento_consulta',
    'departamento_consulta_codigo',
    'municipio_consulta',
]

df = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
df_inferred = pd.read_csv(RAW_PATH)

cat_departamentos = pd.read_csv(CAT_DIR / 'departamentos.csv', dtype=str, keep_default_na=False)
cat_municipios = pd.read_csv(CAT_DIR / 'municipios.csv', dtype=str, keep_default_na=False)
cat_sectores = pd.read_csv(CAT_DIR / 'sectores.csv', dtype=str, keep_default_na=False)
cat_modalidades = pd.read_csv(CAT_DIR / 'modalidades.csv', dtype=str, keep_default_na=False)

valid_departments = set(cat_departamentos['label'])
valid_department_codes = set(cat_departamentos['value'])
valid_municipality_pairs = set(zip(cat_municipios['departamento'], cat_municipios['municipio']))
valid_sector_values = set(cat_sectores['label'])
valid_modalidad_values = set(cat_modalidades['label'])

display(Markdown(f"**Archivo analizado:** `{RAW_PATH}`"))

**Archivo analizado:** `C:\Users\diego\Documents\Universidad\Data Science\DSPR1\data\raw\establecimientos_diversificado_guatemala.csv`

In [2]:
general_summary = pd.DataFrame(
    [
        {'metrica': 'registros', 'valor': int(df.shape[0])},
        {'metrica': 'variables', 'valor': int(df.shape[1])},
        {'metrica': 'duplicados_exactos', 'valor': int(df.duplicated().sum())},
        {'metrica': 'columnas_constantes', 'valor': int((df.nunique(dropna=False) == 1).sum())},
    ]
)
display(general_summary)

,metrica,valor
0,registros,11867
1,variables,26
2,duplicados_exactos,0
3,columnas_constantes,8


In [3]:
profile_rows = []

for column in df.columns:
    series = df[column]
    empty_count = int(series.eq('').sum())
    placeholder_count = int(series.isin(PLACEHOLDERS - {''}).sum())
    unique_count = int(series.nunique(dropna=False))
    inferred_dtype = str(df_inferred[column].dtype)

    observations = []
    if empty_count == len(series):
        observations.append('columna completamente vacia')
    elif unique_count == 1:
        observations.append('columna constante')
    if placeholder_count > 0:
        observations.append('placeholders presentes')
    if empty_count > 0:
        observations.append('tiene vacios exactos')
    if column in METADATA_COLUMNS:
        observations.append('metadata de extraccion')

    profile_rows.append(
        {
            'variable': column,
            'tipo_leido_como_texto': 'object',
            'tipo_inferido_por_pandas': inferred_dtype,
            'registros': int(len(series)),
            'valores_unicos': unique_count,
            'vacios_exactos': empty_count,
            'porcentaje_vacios_exactos': round(empty_count / len(series) * 100, 2),
            'placeholders_detectados': placeholder_count,
            'observaciones': '; '.join(observations),
        }
    )

variable_profile = pd.DataFrame(profile_rows).sort_values(['porcentaje_vacios_exactos', 'valores_unicos'], ascending=[False, True])
display(variable_profile)

,variable,tipo_leido_como_texto,tipo_inferido_por_pandas,registros,valores_unicos,vacios_exactos,porcentaje_vacios_exactos,placeholders_detectados,observaciones
25,municipio_consulta,object,float64,11867,1,11867,100.00,0,columna completamente vacia; tiene vacios exactos; metadata de extraccion
8,director,object,str,11867,5518,1733,14.60,99,placeholders presentes; tiene vacios exactos
6,telefono,object,str,11867,6571,946,7.97,0,tiene vacios exactos
7,supervisor,object,str,11867,1281,535,4.51,0,tiene vacios exactos
1,distrito,object,str,11867,1682,532,4.48,0,tiene vacios exactos
5,direccion,object,str,11867,7440,76,0.64,6,placeholders presentes; tiene vacios exactos
4,establecimiento,object,str,11867,6313,5,0.04,0,tiene vacios exactos
9,nivel,object,str,11867,1,0,0.00,0,columna constante
17,fecha_extraccion,object,str,11867,1,0,0.00,0,columna constante; metadata de extraccion
18,fuente_url,object,str,11867,1,0,0.00,0,columna constante; metadata de extraccion


In [4]:
missing_table = variable_profile[
    ['variable', 'vacios_exactos', 'porcentaje_vacios_exactos', 'placeholders_detectados']
].sort_values(['vacios_exactos', 'placeholders_detectados'], ascending=[False, False])

display(missing_table)

,variable,vacios_exactos,porcentaje_vacios_exactos,placeholders_detectados
25,municipio_consulta,11867,100.00,0
8,director,1733,14.60,99
6,telefono,946,7.97,0
7,supervisor,535,4.51,0
1,distrito,532,4.48,0
5,direccion,76,0.64,6
4,establecimiento,5,0.04,0
9,nivel,0,0.00,0
17,fecha_extraccion,0,0.00,0
18,fuente_url,0,0.00,0


In [5]:
unique_table = variable_profile[['variable', 'valores_unicos']].sort_values('valores_unicos', ascending=False)
display(unique_table)

,variable,valores_unicos
0,codigo,11867
5,direccion,7440
6,telefono,6571
4,establecimiento,6313
8,director,5518
1,distrito,1682
7,supervisor,1281
3,municipio,352
16,departamental,26
23,departamento_consulta,23


In [6]:
categorical_columns = ['departamento', 'municipio', 'nivel', 'sector', 'area', 'status', 'modalidad', 'jornada', 'plan', 'departamental']
top_category_rows = []

for column in categorical_columns:
    counts = df[column].value_counts(dropna=False).head(10)
    for value, count in counts.items():
        top_category_rows.append({'variable': column, 'valor': value, 'frecuencia': int(count)})

top_categories = pd.DataFrame(top_category_rows)
display(top_categories)

,variable,valor,frecuencia
0,departamento,CIUDAD CAPITAL,2161
1,departamento,GUATEMALA,1908
2,departamento,SAN MARCOS,724
3,departamento,ESCUINTLA,708
4,departamento,HUEHUETENANGO,591
5,departamento,QUETZALTENANGO,551
6,departamento,PETEN,516
7,departamento,ALTA VERAPAZ,475
8,departamento,SUCHITEPEQUEZ,437
9,departamento,CHIMALTENANGO,435


In [7]:
exact_duplicates = df[df.duplicated(keep=False)].copy()
duplicate_summary = pd.DataFrame([
    {
        'metrica': 'registros_duplicados_exactos',
        'valor': int(df.duplicated().sum()),
    }
])
display(duplicate_summary)
display(exact_duplicates.head(20))

,metrica,valor
0,registros_duplicados_exactos,0


,codigo,distrito,departamento,municipio,establecimiento,direccion,telefono,supervisor,director,nivel,sector,area,status,modalidad,jornada,plan,departamental,fecha_extraccion,fuente_url,nivel_consulta,sector_consulta,plan_consulta,modalidad_consulta,departamento_consulta,departamento_consulta_codigo,municipio_consulta


In [8]:
invalid_departamentos = sorted(set(df['departamento']) - valid_departments)
invalid_departamento_codigo = sorted(set(df['departamento_consulta_codigo']) - valid_department_codes)
invalid_municipality_rows = df.loc[
    ~df.apply(lambda row: (row['departamento'], row['municipio']) in valid_municipality_pairs, axis=1),
    ['codigo', 'departamento', 'municipio']
].copy()
invalid_sector_values = sorted(set(df['sector']) - valid_sector_values)
invalid_modalidad_values = sorted(set(df['modalidad']) - valid_modalidad_values)

domain_summary = pd.DataFrame([
    {
        'variable': 'departamento',
        'regla': 'Debe pertenecer al catalogo de departamentos extraido del sitio',
        'registros_fuera_de_dominio': int(df['departamento'].isin(invalid_departamentos).sum()),
        'valores_observados': ', '.join(invalid_departamentos) if invalid_departamentos else 'Ninguno',
    },
    {
        'variable': 'departamento_consulta_codigo',
        'regla': 'Debe pertenecer al catalogo de codigos de departamento consultados',
        'registros_fuera_de_dominio': int(df['departamento_consulta_codigo'].isin(invalid_departamento_codigo).sum()),
        'valores_observados': ', '.join(invalid_departamento_codigo) if invalid_departamento_codigo else 'Ninguno',
    },
    {
        'variable': 'municipio',
        'regla': 'El par departamento + municipio debe existir en el catalogo extraido del sitio',
        'registros_fuera_de_dominio': int(len(invalid_municipality_rows)),
        'valores_observados': 'Ver archivo de detalle' if len(invalid_municipality_rows) else 'Ninguno',
    },
    {
        'variable': 'sector',
        'regla': 'Debe pertenecer al catalogo de sectores del sitio',
        'registros_fuera_de_dominio': int(df['sector'].isin(invalid_sector_values).sum()),
        'valores_observados': ', '.join(invalid_sector_values) if invalid_sector_values else 'Ninguno',
    },
    {
        'variable': 'modalidad',
        'regla': 'Debe pertenecer al catalogo de modalidades del sitio',
        'registros_fuera_de_dominio': int(df['modalidad'].isin(invalid_modalidad_values).sum()),
        'valores_observados': ', '.join(invalid_modalidad_values) if invalid_modalidad_values else 'Ninguno',
    },
])

display(domain_summary)
display(invalid_municipality_rows.head(20))

,variable,regla,registros_fuera_de_dominio,valores_observados
0,departamento,Debe pertenecer al catalogo de departamentos extraido del sitio,0,Ninguno
1,departamento_consulta_codigo,Debe pertenecer al catalogo de codigos de departamento consultados,0,Ninguno
2,municipio,El par departamento + municipio debe existir en el catalogo extraido del sitio,0,Ninguno
3,sector,Debe pertenecer al catalogo de sectores del sitio,0,Ninguno
4,modalidad,Debe pertenecer al catalogo de modalidades del sitio,0,Ninguno


,codigo,departamento,municipio


In [9]:
consistency_summary = pd.DataFrame([
    {
        'chequeo': 'departamento == departamento_consulta',
        'registros_inconsistentes': int((df['departamento'] != df['departamento_consulta']).sum()),
    },
    {
        'chequeo': 'nivel == DIVERSIFICADO',
        'registros_inconsistentes': int((df['nivel'] != 'DIVERSIFICADO').sum()),
    },
    {
        'chequeo': 'nivel_consulta == DIVERSIFICADO',
        'registros_inconsistentes': int((df['nivel_consulta'] != 'DIVERSIFICADO').sum()),
    },
    {
        'chequeo': 'municipio_consulta vacio por consulta departamental',
        'registros_inconsistentes': int((df['municipio_consulta'] != '').sum()),
    },
])

departamental_mismatches = df.loc[
    df['departamental'] != df['departamento'],
    ['codigo', 'departamento', 'municipio', 'departamental']
].copy()

display(consistency_summary)
display(departamental_mismatches.head(20))

,chequeo,registros_inconsistentes
0,departamento == departamento_consulta,0
1,nivel == DIVERSIFICADO,0
2,nivel_consulta == DIVERSIFICADO,0
3,municipio_consulta vacio por consulta departamental,0


,codigo,departamento,municipio,departamental
1320,00-01-0158-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1321,00-01-0160-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1322,00-01-0162-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1323,00-01-0168-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1324,00-01-0173-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1325,00-01-0174-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1326,00-01-0175-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1327,00-01-0176-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1328,00-01-0177-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE
1329,00-01-0178-46,CIUDAD CAPITAL,ZONA 1,GUATEMALA NORTE


In [10]:
def classify_distrito(value: str) -> str:
    if value == '':
        return 'vacio'
    if re.fullmatch(r'\d{2}-\d{3}', value):
        return 'NN-NNN'
    if re.fullmatch(r'\d{2}-\d{2}-\d{3}', value):
        return 'NN-NN-NNN'
    if re.fullmatch(r'\d{2}-\d{2}-\d{4}', value):
        return 'NN-NN-NNNN'
    return 'otro'

telefono_non_empty = df['telefono'].ne('')
telefono_non_digit_rows = df.loc[
    telefono_non_empty & ~df['telefono'].str.fullmatch(r'\d+'),
    ['codigo', 'telefono']
].copy()
telefono_length_distribution = (
    df.loc[telefono_non_empty, 'telefono']
    .str.len()
    .value_counts()
    .sort_index()
    .rename_axis('longitud')
    .reset_index(name='frecuencia')
)
codigo_bad_format = df.loc[~df['codigo'].str.fullmatch(r'\d{2}-\d{2}-\d{4}-\d{2}'), ['codigo']].copy()
distrito_patterns = df['distrito'].map(classify_distrito).value_counts().rename_axis('patron').reset_index(name='frecuencia')
distrito_other_rows = df.loc[df['distrito'].map(classify_distrito) == 'otro', ['codigo', 'distrito']].copy()

format_summary = pd.DataFrame([
    {
        'variable': 'telefono',
        'chequeo': 'Debe contener solo digitos cuando no esta vacio',
        'registros_inconsistentes': int(len(telefono_non_digit_rows)),
    },
    {
        'variable': 'codigo',
        'chequeo': 'Debe seguir el patron NN-NN-NNNN-NN',
        'registros_inconsistentes': int(len(codigo_bad_format)),
    },
    {
        'variable': 'distrito',
        'chequeo': 'Se reportan patrones observados; los valores en categoria otro requieren revision',
        'registros_inconsistentes': int(len(distrito_other_rows)),
    },
])


display(format_summary)
display(telefono_length_distribution)
display(distrito_patterns)

,variable,chequeo,registros_inconsistentes
0,telefono,Debe contener solo digitos cuando no esta vacio,201
1,codigo,Debe seguir el patron NN-NN-NNNN-NN,0
2,distrito,Se reportan patrones observados; los valores en categoria otro requieren revision,70


,longitud,frecuencia
0,2,1
1,4,1
2,5,4
3,6,8
4,7,34
5,8,10671
6,9,4
7,10,2
8,11,11
9,13,2


,patron,frecuencia
0,NN-NN-NNNN,6226
1,NN-NNN,5039
2,vacio,532
3,otro,70


In [11]:
quality_issues = pd.DataFrame([
    {
        'variable': 'municipio_consulta',
        'tipo_problema': 'columna completamente vacia',
        'evidencia': int(df['municipio_consulta'].eq('').sum()),
        'detalle': 'No aporta informacion porque la consulta fue a nivel de departamento',
    },
    {
        'variable': 'fecha_extraccion, fuente_url, nivel_consulta, sector_consulta, plan_consulta, modalidad_consulta',
        'tipo_problema': 'columnas constantes',
        'evidencia': int((df[[
            'fecha_extraccion', 'fuente_url', 'nivel_consulta', 'sector_consulta', 'plan_consulta', 'modalidad_consulta'
        ]].nunique(dropna=False) == 1).sum()),
        'detalle': 'Tienen valor documental, pero muy poco valor analitico para modelado o estadistica descriptiva',
    },
    {
        'variable': 'telefono',
        'tipo_problema': 'formato inconsistente',
        'evidencia': int(len(telefono_non_digit_rows)),
        'detalle': 'Ademas de vacios, aparecen longitudes heterogeneas y valores no puramente numericos',
    },
    {
        'variable': 'distrito',
        'tipo_problema': 'patrones mixtos',
        'evidencia': int(len(distrito_other_rows)),
        'detalle': 'Existen varios formatos observados; se requiere definir regla de estandarizacion en limpieza',
    },
    {
        'variable': 'director',
        'tipo_problema': 'faltantes y placeholders',
        'evidencia': int(df['director'].eq('').sum() + df['director'].eq('--').sum()),
        'detalle': 'Combina vacios exactos con placeholders que deben tratarse de forma consistente',
    },
    {
        'variable': 'departamental',
        'tipo_problema': 'categoria administrativa distinta de departamento',
        'evidencia': int((df['departamental'] != df['departamento']).sum()),
        'detalle': 'No debe unificarse automaticamente con departamento; primero debe definirse su semantica',
    },
    {
        'variable': 'establecimiento',
        'tipo_problema': 'posibles duplicados parciales futuros',
        'evidencia': int(df['establecimiento'].duplicated(keep=False).sum()),
        'detalle': 'El nombre del establecimiento se repite en muchos codigos distintos; esto requiere analisis posterior por similitud y contexto',
    },
])

display(quality_issues)

,variable,tipo_problema,evidencia,detalle
0,municipio_consulta,columna completamente vacia,11867,No aporta informacion porque la consulta fue a nivel de departamento
1,"fecha_extraccion, fuente_url, nivel_consulta, sector_consulta, plan_consulta, modalidad_consulta",columnas constantes,6,"Tienen valor documental, pero muy poco valor analitico para modelado o estadistica descriptiva"
2,telefono,formato inconsistente,201,"Ademas de vacios, aparecen longitudes heterogeneas y valores no puramente numericos"
3,distrito,patrones mixtos,70,Existen varios formatos observados; se requiere definir regla de estandarizacion en limpieza
4,director,faltantes y placeholders,1774,Combina vacios exactos con placeholders que deben tratarse de forma consistente
5,departamental,categoria administrativa distinta de departamento,6094,No debe unificarse automaticamente con departamento; primero debe definirse su semantica
6,establecimiento,posibles duplicados parciales futuros,7741,El nombre del establecimiento se repite en muchos codigos distintos; esto requiere analisis posterior por similitud ...


In [12]:
hallazgos_clave = pd.DataFrame([
    {'hallazgo': 'Registros', 'valor': int(df.shape[0]), 'comentario': 'Volumen total del dataset crudo'},
    {'hallazgo': 'Variables', 'valor': int(df.shape[1]), 'comentario': 'Incluye variables de negocio y metadatos de extraccion'},
    {'hallazgo': 'Duplicados exactos', 'valor': int(df.duplicated().sum()), 'comentario': 'No se detectaron filas identicas en todas las columnas'},
    {'hallazgo': 'Telefonos vacios', 'valor': int(df['telefono'].eq('').sum()), 'comentario': 'Faltante relevante para contacto'},
    {'hallazgo': 'Telefonos no numericos', 'valor': int(len(telefono_non_digit_rows)), 'comentario': 'Formato inconsistente'},
    {'hallazgo': 'Directores vacios o placeholder', 'valor': int(df['director'].eq('').sum() + df['director'].eq('--').sum()), 'comentario': 'Combina faltantes y marcador textual'},
    {'hallazgo': 'Departamental distinto a departamento', 'valor': int((df['departamental'] != df['departamento']).sum()), 'comentario': 'Sugiere una estructura administrativa distinta'},
    {'hallazgo': 'Municipio fuera de catalogo', 'valor': int(len(invalid_municipality_rows)), 'comentario': 'No se detectaron pares invalidos en la exploracion automatica'},
])

display(hallazgos_clave)

display(Markdown('### Cierre del diagnóstico'))
display(Markdown(
    'El conjunto crudo es estructuralmente usable, pero presenta problemas de faltantes, placeholders, formatos mixtos y variables administrativas cuyo significado debe preservarse antes de cualquier limpieza agresiva. Estos hallazgos respaldan la siguiente fase: elaborar el plan de limpieza variable por variable.'
))

,hallazgo,valor,comentario
0,Registros,11867,Volumen total del dataset crudo
1,Variables,26,Incluye variables de negocio y metadatos de extraccion
2,Duplicados exactos,0,No se detectaron filas identicas en todas las columnas
3,Telefonos vacios,946,Faltante relevante para contacto
4,Telefonos no numericos,201,Formato inconsistente
5,Directores vacios o placeholder,1774,Combina faltantes y marcador textual
6,Departamental distinto a departamento,6094,Sugiere una estructura administrativa distinta
7,Municipio fuera de catalogo,0,No se detectaron pares invalidos en la exploracion automatica


### Cierre del diagnóstico

El conjunto crudo es estructuralmente usable, pero presenta problemas de faltantes, placeholders, formatos mixtos y variables administrativas cuyo significado debe preservarse antes de cualquier limpieza agresiva. Estos hallazgos respaldan la siguiente fase: elaborar el plan de limpieza variable por variable.